# 📚 Research Literature Review Agent

**Automatically generate a publication-quality literature review from arXiv papers.**

```
User Query → arXiv Search → Embedding + FAISS → LLM Summarization → Synthesis → Review
```

---
### 🚦 Quick Start (3 steps)
1. Add your OpenAI API key to **Colab Secrets** — click the 🔑 icon in the left sidebar  
   &nbsp;&nbsp;&nbsp;Name: `OPENAI_API_KEY` &nbsp;&nbsp;Value: `sk-...`
2. Click **Runtime → Run all** (or press `Ctrl+F9`)
3. Wait ~3 minutes — your literature review will appear at the bottom

> **This notebook is fully self-contained.** All source code is written to disk automatically.  
> No GitHub clone needed. Just run and it works.


---
## Step 1 — Install Dependencies
*Takes ~60 seconds on first run. Safe to re-run.*

In [ ]:
!pip install -q \
    langchain==0.3.25 \
    langchain-community==0.3.24 \
    langchain-openai==0.3.16 \
    langchain-core==0.3.57 \
    openai>=1.30.0 \
    arxiv>=2.1.0 \
    faiss-cpu>=1.7.4 \
    tiktoken>=0.7.0 \
    python-dotenv>=1.0.0

print("✅ All packages installed successfully.")


---
## Step 2 — Load API Key

**Use Colab Secrets (recommended — your key stays private):**
1. Click the 🔑 icon in the left sidebar
2. Click **"Add new secret"**
3. Name: `OPENAI_API_KEY` | Value: `sk-your-key-here`
4. Toggle **"Notebook access"** to ON
5. Run this cell


In [ ]:
import os

# ── Option A: Colab Secrets (recommended) ─────────────────────────────────────
try:
    from google.colab import userdata
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
    print("✅ API key loaded from Colab Secrets.")
except Exception as e:
    print(f"⚠  Colab Secrets unavailable ({e}).")
    # ── Option B: Paste directly (only use on a private/local machine) ──────────
    # os.environ['OPENAI_API_KEY'] = "sk-..."   # ← uncomment and paste key here
    print("   If you see this, uncomment Option B above and paste your key.")

# Verify the key is set
key = os.environ.get('OPENAI_API_KEY', '')
if key.startswith('sk-') and len(key) > 20:
    print(f"✅ Key detected: {key[:8]}...{key[-4:]}")
else:
    print("❌ Key not found or invalid. Please set it before continuing.")


---
## Step 3 — Write Source Files to Disk

This step writes the entire `research_agent` Python package to Colab's filesystem.  
You don't need to edit anything here — just run it.


In [ ]:
import os
os.makedirs('research_agent', exist_ok=True)
print('📁 Package directory created.')


In [ ]:
%%writefile research_agent/__init__.py
"""research_agent package"""
from research_agent.arxiv_retriever import ArxivRetriever
from research_agent.vector_store import VectorStore
from research_agent.summarizer import Summarizer
from research_agent.literature_generator import LiteratureGenerator
from research_agent.agent import run_literature_review_agent
from research_agent.config import Config

__version__ = "1.0.0"
__all__ = ["run_literature_review_agent","ArxivRetriever","VectorStore",
           "Summarizer","LiteratureGenerator","Config"]


In [ ]:
%%writefile research_agent/config.py
"""
research_agent/config.py
------------------------
Centralised configuration for the Literature Review Agent.

All tuneable parameters (model names, chunk sizes, API keys, output paths)
live here so nothing is scattered across modules. Change behaviour by editing
this file or by passing a Config object directly to any pipeline component.

Environment variables always take precedence over default values.
"""

import os
from dataclasses import dataclass, field
from pathlib import Path
from dotenv import load_dotenv

# Load .env if present (harmless if absent)
load_dotenv()


@dataclass
class Config:
    """
    Holds all runtime configuration for the agent.

    Attributes
    ----------
    openai_api_key : str
        Your OpenAI API key. Reads from OPENAI_API_KEY env var by default.
    summarization_model : str
        The LLM used for per-paper summarization.
        Default: "gpt-4o-mini" (fast and cost-efficient).
    synthesis_model : str
        The LLM used for the final literature review synthesis.
        Default: "gpt-4o" (higher quality for the critical final step).
    embedding_model : str
        OpenAI embedding model for vectorising paper text.
    max_papers : int
        Default number of arXiv papers to retrieve.
    chunk_size : int
        Character size of each text chunk before embedding.
    chunk_overlap : int
        Character overlap between adjacent chunks (preserves sentence continuity).
    retrieval_k : int
        Number of vector-store chunks returned per similarity query.
    summarization_temperature : float
        LLM temperature for summarization (lower = more factual).
    synthesis_temperature : float
        LLM temperature for synthesis (slightly higher for fluent prose).
    output_dir : Path
        Directory where generated reviews are saved.
    save_output : bool
        Whether to save the final review to disk automatically.
    arxiv_sort_by : str
        arXiv sort criterion: "relevance" | "lastUpdatedDate" | "submittedDate".
    """

    # ── API credentials ───────────────────────────────────────────────────────
    openai_api_key: str = field(
        default_factory=lambda: os.environ.get("OPENAI_API_KEY", "")
    )

    # ── Model selection ───────────────────────────────────────────────────────
    summarization_model: str = field(
        default_factory=lambda: os.environ.get("SUMMARIZATION_MODEL", "gpt-4o-mini")
    )
    synthesis_model: str = field(
        default_factory=lambda: os.environ.get("SYNTHESIS_MODEL", "gpt-4o")
    )
    embedding_model: str = field(
        default_factory=lambda: os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")
    )

    # ── Retrieval settings ────────────────────────────────────────────────────
    max_papers: int = field(
        default_factory=lambda: int(os.environ.get("MAX_PAPERS", "8"))
    )
    arxiv_sort_by: str = "relevance"

    # ── Chunking settings ─────────────────────────────────────────────────────
    chunk_size: int = 800
    chunk_overlap: int = 100

    # ── Vector store retrieval ────────────────────────────────────────────────
    retrieval_k: int = 6

    # ── LLM generation settings ───────────────────────────────────────────────
    summarization_temperature: float = 0.2
    synthesis_temperature: float = 0.3

    # ── Output settings ───────────────────────────────────────────────────────
    output_dir: Path = field(default_factory=lambda: Path("outputs"))
    save_output: bool = True

    def validate(self) -> None:
        """
        Raise a clear error if required configuration is missing.
        Call this at agent startup before any API calls are made.
        """
        if not self.openai_api_key:
            raise ValueError(
                "\n[Config] OpenAI API key is missing.\n"
                "  Set it with:  export OPENAI_API_KEY='sk-...'\n"
                "  Or create a .env file with OPENAI_API_KEY=sk-...\n"
                "  Or pass it directly: Config(openai_api_key='sk-...')"
            )
        if self.max_papers < 1 or self.max_papers > 50:
            raise ValueError("max_papers must be between 1 and 50.")
        if self.chunk_size < 200:
            raise ValueError("chunk_size must be at least 200 characters.")

    def __post_init__(self) -> None:
        """Ensure output_dir is always a Path object."""
        self.output_dir = Path(self.output_dir)


In [ ]:
%%writefile research_agent/arxiv_retriever.py
"""
research_agent/arxiv_retriever.py
----------------------------------
Handles all communication with the arXiv API.

HOW IT WORKS
------------
The arXiv API is a public REST service provided by Cornell University.
This module wraps it using the `arxiv` Python library, which handles
rate limiting and pagination automatically.

When you call ArxivRetriever.search(), the library sends an HTTP request to:
  https://export.arxiv.org/api/query?search_query=...

Results are returned as Python objects with attributes like .title,
.authors, .summary (abstract), .published, and .entry_id.

We convert each result into a plain Python dict for portability — no
dependency on the arxiv library's internal objects elsewhere in the codebase.

SCALING NOTE
------------
arXiv imposes a soft rate limit of ~3 requests/second. For large-scale
scraping (>100 papers), add time.sleep(1) between batches. For this
agent's default of 5–15 papers, no throttling is needed.
"""

import arxiv
from typing import List, Dict
from research_agent.config import Config


class ArxivRetriever:
    """
    Searches arXiv for papers matching a topic query and returns
    structured metadata dictionaries.

    Parameters
    ----------
    config : Config
        Global agent configuration. Uses config.max_papers and
        config.arxiv_sort_by.

    Example
    -------
    >>> retriever = ArxivRetriever(config)
    >>> papers = retriever.search("multi-agent reinforcement learning", max_results=6)
    >>> print(papers[0]["title"])
    """

    # Map human-readable sort names to arxiv library constants
    SORT_OPTIONS = {
        "relevance":       arxiv.SortCriterion.Relevance,
        "lastUpdatedDate": arxiv.SortCriterion.LastUpdatedDate,
        "submittedDate":   arxiv.SortCriterion.SubmittedDate,
    }

    def __init__(self, config: Config) -> None:
        self.config = config
        sort_key = config.arxiv_sort_by
        self.sort_criterion = self.SORT_OPTIONS.get(
            sort_key, arxiv.SortCriterion.Relevance
        )

    def search(self, query: str, max_results: int = None) -> List[Dict]:
        """
        Search arXiv and return a list of paper metadata dicts.

        Parameters
        ----------
        query       : Natural language research topic.
        max_results : Number of papers to fetch. Defaults to config.max_papers.

        Returns
        -------
        List of dicts with keys:
            title, authors, abstract, url, date, paper_id
        """
        if max_results is None:
            max_results = self.config.max_papers

        print(f"\n[ArxivRetriever] Searching: '{query}'  (max {max_results} papers)")

        search_obj = arxiv.Search(
            query=query,
            max_results=max_results,
            sort_by=self.sort_criterion,
        )

        papers = []
        for result in search_obj.results():
            paper = {
                "title":    result.title,
                # Convert Author objects to plain strings
                "authors":  [str(a) for a in result.authors],
                # Flatten newlines so the text is one clean paragraph
                "abstract": result.summary.replace("\n", " ").strip(),
                "url":      result.entry_id,
                "date":     str(result.published.date()),
                # Extract just the numeric ID from the full URL
                "paper_id": result.entry_id.split("/abs/")[-1],
            }
            papers.append(paper)
            print(f"  ✓ {paper['title'][:80]}...")

        print(f"[ArxivRetriever] Retrieved {len(papers)} papers.\n")
        return papers

    @staticmethod
    def to_plain_text(papers: List[Dict]) -> List[str]:
        """
        Convert paper metadata dicts into plain-text strings.

        Each string includes title, authors, date, URL, and abstract —
        formatted so the LLM can read it naturally.

        Parameters
        ----------
        papers : Output of ArxivRetriever.search()

        Returns
        -------
        List of formatted strings, one per paper.
        """
        docs = []
        for p in papers:
            # Limit author list to first 5 to keep text concise
            author_list = p["authors"][:5]
            if len(p["authors"]) > 5:
                author_list.append("et al.")
            authors_str = ", ".join(author_list)

            text = (
                f"TITLE: {p['title']}\n"
                f"AUTHORS: {authors_str}\n"
                f"DATE: {p['date']}\n"
                f"URL: {p['url']}\n"
                f"ABSTRACT: {p['abstract']}"
            )
            docs.append(text)
        return docs


In [ ]:
%%writefile research_agent/vector_store.py
"""
research_agent/vector_store.py
-------------------------------
Converts paper text into dense vector embeddings and stores them in a
FAISS in-memory index for fast semantic similarity search.

WHAT IS AN EMBEDDING?
---------------------
An embedding model takes a piece of text and outputs a list of numbers
(a "vector"). Similar texts produce vectors that are numerically close.
For example, "weather forecasting with neural networks" and "deep learning
for atmospheric prediction" will have very similar vectors even though they
share no exact words.

WHAT IS FAISS?
--------------
FAISS (Facebook AI Similarity Search) is a library that stores millions of
vectors and finds the closest ones to a query vector in milliseconds. It is
used here in "flat" (exact search) mode since our dataset is small (<200 chunks).
For larger datasets, switch to an IVF or HNSW index.

WHAT IS TEXT SPLITTING?
-----------------------
LLMs and embedding models have maximum input lengths (context windows).
A paper abstract + title might be ~500 words — fine on its own. But to
handle longer PDFs in the future, we split every document into overlapping
chunks of `chunk_size` characters with `chunk_overlap` characters of overlap.
The overlap ensures no sentence is cut off at a boundary without its neighbour.

PIPELINE
--------
  List[str] (paper texts)
      → RecursiveCharacterTextSplitter → List[Document] (chunks)
      → OpenAIEmbeddings                → List[List[float]] (vectors)
      → FAISS.from_documents            → FAISS index
      → similarity_search(query, k)     → List[Document] (most relevant chunks)
"""

from typing import List
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain.schema import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

from research_agent.config import Config


class VectorStore:
    """
    Wraps the full embedding + FAISS pipeline into a single reusable object.

    Parameters
    ----------
    config : Config
        Uses config.embedding_model, config.chunk_size,
        config.chunk_overlap, config.retrieval_k, config.openai_api_key.

    Example
    -------
    >>> vs = VectorStore(config)
    >>> vs.build(doc_texts)
    >>> chunks = vs.retrieve("deep learning precipitation", k=5)
    """

    def __init__(self, config: Config) -> None:
        self.config = config
        self._index: FAISS = None  # populated by .build()

        # ── Embedding model ──────────────────────────────────────────────────
        # text-embedding-3-small: 1536-dimensional vectors, fast, low cost.
        # text-embedding-3-large: higher quality, 3072-dimensional, ~6× more expensive.
        self.embeddings = OpenAIEmbeddings(
            model=config.embedding_model,
            openai_api_key=config.openai_api_key,
        )

        # ── Text splitter ────────────────────────────────────────────────────
        # RecursiveCharacterTextSplitter tries to split at paragraph → sentence
        # → word boundaries in that order, so splits are semantically cleaner
        # than a naive fixed-character split.
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=config.chunk_size,
            chunk_overlap=config.chunk_overlap,
            length_function=len,
        )

    # ── Public methods ────────────────────────────────────────────────────────

    def build(self, texts: List[str]) -> None:
        """
        Chunk, embed, and index a list of plain-text paper strings.

        Parameters
        ----------
        texts : Output of ArxivRetriever.to_plain_text()
        """
        # Wrap each text string in a LangChain Document object.
        # Document pairs text with optional metadata (source, page, etc).
        documents = [Document(page_content=t) for t in texts]

        # Split into chunks
        chunks = self.splitter.split_documents(documents)
        print(f"[VectorStore] {len(documents)} documents → {len(chunks)} chunks.")

        # Embed and index — this makes one OpenAI API call per batch of chunks
        self._index = FAISS.from_documents(chunks, self.embeddings)
        print(f"[VectorStore] FAISS index ready.\n")

    def retrieve(self, query: str, k: int = None) -> List[str]:
        """
        Return the k most semantically similar chunks to a query string.

        Parameters
        ----------
        query : The search query (typically the research topic).
        k     : Number of chunks to return. Defaults to config.retrieval_k.

        Returns
        -------
        List of raw text strings from the most relevant chunks.
        """
        if self._index is None:
            raise RuntimeError("Call VectorStore.build() before retrieve().")
        if k is None:
            k = self.config.retrieval_k

        results = self._index.similarity_search(query, k=k)
        return [doc.page_content for doc in results]

    def save(self, path: str) -> None:
        """Persist the FAISS index to disk for reuse across sessions."""
        if self._index is None:
            raise RuntimeError("No index to save. Call build() first.")
        self._index.save_local(path)
        print(f"[VectorStore] Index saved to: {path}")

    def load(self, path: str) -> None:
        """Load a previously saved FAISS index from disk."""
        self._index = FAISS.load_local(
            path,
            self.embeddings,
            allow_dangerous_deserialization=True,
        )
        print(f"[VectorStore] Index loaded from: {path}")


In [ ]:
%%writefile research_agent/utils.py
"""
research_agent/utils.py
------------------------
Shared utility functions used across the agent pipeline.

Contains helpers for:
  - Saving reviews to disk (plain text and LaTeX)
  - Pretty-printing pipeline progress
  - Sanitising strings for use as filenames
  - Timing pipeline stages
"""

import os
import time
import re
from pathlib import Path
from datetime import datetime
from typing import Optional
from contextlib import contextmanager


# ── File I/O ──────────────────────────────────────────────────────────────────

def save_review(
    content: str,
    topic: str,
    output_dir: Path,
    extension: str = "txt",
) -> Path:
    """
    Save a generated review to disk with a timestamp-based filename.

    Parameters
    ----------
    content    : Text content to write.
    topic      : Research topic (used in filename).
    output_dir : Directory to save into (created if it does not exist).
    extension  : File extension, e.g. "txt" or "tex".

    Returns
    -------
    Path to the saved file.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    safe_topic = sanitise_filename(topic, max_length=40)
    timestamp  = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename   = output_dir / f"review_{safe_topic}_{timestamp}.{extension}"

    with open(filename, "w", encoding="utf-8") as f:
        f.write(content)

    print(f"[Utils] Saved → {filename}")
    return filename


def sanitise_filename(text: str, max_length: int = 50) -> str:
    """
    Convert an arbitrary string into a safe, lowercase filename fragment.

    Replaces spaces with underscores, strips non-alphanumeric characters
    (except underscores and hyphens), and truncates to max_length.

    Examples
    --------
    >>> sanitise_filename("Agentic AI: Climate & Prediction!")
    'agentic_ai_climate__prediction'
    """
    text = text.lower()
    text = re.sub(r"[^\w\s-]", "", text)      # remove special chars
    text = re.sub(r"[\s]+", "_", text)         # spaces → underscores
    return text[:max_length]


# ── Timing ────────────────────────────────────────────────────────────────────

@contextmanager
def timer(label: str):
    """
    Context manager that prints elapsed time for a code block.

    Usage
    -----
    >>> with timer("ArXiv retrieval"):
    ...     papers = retriever.search(topic)
    [Timer] ArXiv retrieval — 1.84s
    """
    start = time.perf_counter()
    yield
    elapsed = time.perf_counter() - start
    print(f"[Timer] {label} — {elapsed:.2f}s")


# ── Display helpers ───────────────────────────────────────────────────────────

def print_banner(title: str, width: int = 60) -> None:
    """Print a formatted banner line for pipeline stage announcements."""
    border = "─" * width
    print(f"\n┌{border}┐")
    print(f"│  {title:<{width - 2}}│")
    print(f"└{border}┘")


def print_paper_table(papers: list) -> None:
    """
    Print a compact table of retrieved papers for quick review.

    Parameters
    ----------
    papers : List of paper dicts from ArxivRetriever.search()
    """
    print(f"\n{'#':<4} {'Date':<12} {'Title':<60}")
    print("─" * 78)
    for i, p in enumerate(papers, 1):
        title_short = p["title"][:58] + ".." if len(p["title"]) > 60 else p["title"]
        print(f"{i:<4} {p['date']:<12} {title_short}")
    print()


# ── Validation helpers ────────────────────────────────────────────────────────

def validate_papers(papers: list, min_count: int = 2) -> None:
    """
    Raise a clear error if the paper list is too small to synthesize from.

    Parameters
    ----------
    papers    : List of paper dicts.
    min_count : Minimum number of papers required (default: 2).
    """
    if not papers:
        raise ValueError(
            "No papers were retrieved. Try a broader search query, "
            "or check your internet connection."
        )
    if len(papers) < min_count:
        raise ValueError(
            f"Only {len(papers)} paper(s) retrieved. At least {min_count} are "
            "needed for a meaningful literature review. Increase max_papers or "
            "broaden the search topic."
        )


In [ ]:
%%writefile research_agent/summarizer.py
"""
research_agent/summarizer.py
-----------------------------
Per-paper LLM summarization using the modern LangChain LCEL pipe syntax.
"""
from typing import List, Dict
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from research_agent.config import Config

SUMMARIZATION_PROMPT = PromptTemplate(
    input_variables=["title", "abstract"],
    template="""You are a research assistant writing structured summaries for an academic literature review.

Given the paper title and abstract below, write a concise 2-4 sentence summary that covers:
1. Core research problem or motivation
2. Proposed method, model, or framework
3. Key results or contributions
4. Limitations if explicitly mentioned (omit if not)

Rules:
- Use formal academic English
- Be precise and specific — include model names, dataset names, and metrics where present
- Do not pad with generic phrases like "This paper presents..."
- Write as flowing prose, not bullet points

TITLE: {title}
ABSTRACT: {abstract}

SUMMARY:""",
)


class Summarizer:
    """Uses an LLM to produce structured per-paper summaries."""

    def __init__(self, config: Config) -> None:
        self.config = config
        llm = ChatOpenAI(
            model=config.summarization_model,
            temperature=config.summarization_temperature,
            openai_api_key=config.openai_api_key,
        )
        # Modern LCEL chain: prompt | llm | output_parser
        self._chain = SUMMARIZATION_PROMPT | llm | StrOutputParser()

    def summarize_paper(self, paper: Dict) -> str:
        result = self._chain.invoke({
            "title":    paper["title"],
            "abstract": paper["abstract"],
        })
        return result.strip()

    def summarize_all(self, papers: List[Dict]) -> List[Dict]:
        print(f"[Summarizer] Summarizing {len(papers)} papers...")
        for i, paper in enumerate(papers, 1):
            print(f"  [{i}/{len(papers)}] {paper['title'][:70]}...")
            paper["summary"] = self.summarize_paper(paper)
        print("[Summarizer] Done.\n")
        return papers


In [ ]:
%%writefile research_agent/literature_generator.py
"""
research_agent/literature_generator.py
----------------------------------------
Literature synthesis and output formatting.
"""
from typing import List, Dict
from langchain_openai import ChatOpenAI
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from research_agent.config import Config

SYNTHESIS_PROMPT = PromptTemplate(
    input_variables=["topic", "num_papers", "summaries"],
    template="""You are a senior AI researcher writing the literature review section of a research paper.

Research Topic: {topic}
Number of papers analyzed: {num_papers}

Below are structured summaries of each paper:

{summaries}

Write a literature review of exactly 3 paragraphs following this structure:

PARAGRAPH 1 — Research Trends & Context
Describe the overall research trajectory in this area. What paradigms or approaches have been dominant? How has the field evolved?

PARAGRAPH 2 — Methods & Key Contributions
Synthesize the major methodological approaches, architectures, and frameworks. Highlight landmark results. Reference papers by title or lead author where appropriate.

PARAGRAPH 3 — Research Gaps & Future Directions
Identify what is missing, under-explored, or explicitly stated as a limitation. Frame these gaps to motivate future work.

Formatting rules:
- Write in formal, academic English
- Use flowing paragraphs — no bullet points, no headings
- Be specific: name models, datasets, metrics where present
- Do not include anything outside the three paragraphs

LITERATURE REVIEW:""",
)


class LiteratureGenerator:
    """Synthesizes per-paper summaries into a cohesive literature review."""

    def __init__(self, config: Config) -> None:
        self.config = config
        llm = ChatOpenAI(
            model=config.synthesis_model,
            temperature=config.synthesis_temperature,
            openai_api_key=config.openai_api_key,
        )
        self._chain = SYNTHESIS_PROMPT | llm | StrOutputParser()

    def generate(self, topic: str, papers: List[Dict]) -> str:
        print("[LiteratureGenerator] Synthesizing review...\n")
        summaries_block = ""
        for i, p in enumerate(papers, 1):
            summaries_block += f"[{i}] {p['title']}\n{p['summary']}\n\n"
        result = self._chain.invoke({
            "topic":      topic,
            "num_papers": str(len(papers)),
            "summaries":  summaries_block,
        })
        print("[LiteratureGenerator] Done.\n")
        return result.strip()

    @staticmethod
    def format_output(topic: str, papers: List[Dict], review: str) -> str:
        sep = "=" * 72
        refs = ""
        for i, p in enumerate(papers, 1):
            authors = p["authors"][:3]
            if len(p["authors"]) > 3:
                authors = authors + ["et al."]
            year = p["date"][:4]
            refs += f"  [{i}] {', '.join(authors)} ({year}). {p['title']}.\n       {p['url']}\n"
        return (
            f"\n{sep}\n"
            f"  AUTOMATED LITERATURE REVIEW\n"
            f"  Topic    : {topic}\n"
            f"  Papers   : {len(papers)}\n"
            f"{sep}\n\n"
            f"{review}\n\n"
            f"{sep}\n  REFERENCES\n{sep}\n"
            f"{refs}{sep}\n"
        )

    @staticmethod
    def to_latex(topic: str, papers: List[Dict], review: str) -> str:
        escape = {"&": r"\&", "%": r"\%", "$": r"\$", "#": r"\#",
                  "_": r"\_", "{": r"\{", "}": r"\}"}
        safe = review
        for c, r in escape.items():
            safe = safe.replace(c, r)
        bibitems = ""
        for i, p in enumerate(papers, 1):
            bibitems += (
                f"\\bibitem{{ref{i}}}\n"
                f"  {', '.join(p['authors'][:3])} ({p['date'][:4]}).\n"
                f"  \\textit{{{p['title']}}}.\n"
                f"  \\url{{{p['url']}}}\n\n"
            )
        return (
            f"% Generated by Research Literature Review Agent\n"
            f"\\section{{Related Work}}\n\n{safe}\n\n"
            f"\\begin{{thebibliography}}{{99}}\n{bibitems}\\end{{thebibliography}}\n"
        )


In [ ]:
%%writefile research_agent/agent.py
"""
research_agent/agent.py
------------------------
Pipeline orchestration.
"""
import argparse
from pathlib import Path
from typing import Optional

from research_agent.config import Config
from research_agent.arxiv_retriever import ArxivRetriever
from research_agent.vector_store import VectorStore
from research_agent.summarizer import Summarizer
from research_agent.literature_generator import LiteratureGenerator
from research_agent.utils import save_review, print_banner, print_paper_table, validate_papers, timer


def run_literature_review_agent(
    topic: str,
    max_papers: int = 8,
    openai_api_key: Optional[str] = None,
    save_txt: bool = True,
    save_latex: bool = False,
    output_dir: str = "outputs",
    config: Optional[Config] = None,
) -> str:
    if config is None:
        config = Config(max_papers=max_papers, output_dir=Path(output_dir), save_output=save_txt)
        if openai_api_key:
            config.openai_api_key = openai_api_key
    config.validate()

    print_banner(f"LITERATURE REVIEW AGENT  |  \"{topic}\"")

    with timer("ArXiv retrieval"):
        retriever = ArxivRetriever(config)
        papers = retriever.search(topic, max_results=config.max_papers)
    validate_papers(papers, min_count=2)
    print_paper_table(papers)

    with timer("Embedding + indexing"):
        doc_texts = ArxivRetriever.to_plain_text(papers)
        vs = VectorStore(config)
        vs.build(doc_texts)
    top_chunks = vs.retrieve(topic)
    print(f"[Agent] Top {len(top_chunks)} chunks retrieved.\n")

    with timer("Summarization"):
        summarizer = Summarizer(config)
        papers = summarizer.summarize_all(papers)

    with timer("Synthesis"):
        generator = LiteratureGenerator(config)
        review_text = generator.generate(topic, papers)

    final_output = LiteratureGenerator.format_output(topic, papers, review_text)
    print(final_output)

    if save_txt:
        save_review(final_output, topic, config.output_dir, extension="txt")
    if save_latex:
        latex_output = LiteratureGenerator.to_latex(topic, papers, review_text)
        save_review(latex_output, topic, config.output_dir, extension="tex")

    return final_output


In [ ]:
import os
required = [
    "research_agent/__init__.py",
    "research_agent/config.py",
    "research_agent/arxiv_retriever.py",
    "research_agent/vector_store.py",
    "research_agent/summarizer.py",
    "research_agent/literature_generator.py",
    "research_agent/utils.py",
    "research_agent/agent.py",
]
all_ok = True
for f in required:
    exists = os.path.exists(f)
    status = "✅" if exists else "❌"
    if not exists:
        all_ok = False
    print(f"{status} {f}")

if all_ok:
    print("\n✅ All source files written. Ready to run.")
else:
    print("\n❌ Some files missing — re-run Step 3 cells.")


---
## Step 4 — Configure Your Search & Run the Agent

⚙️ **Edit `TOPIC` and `MAX_PAPERS` below, then run this cell.**

| Setting | Recommended value | Notes |
|---------|------------------|-------|
| `TOPIC` | Any research subject | Be specific for better results |
| `MAX_PAPERS` | 5–10 | More papers = richer review, higher cost |
| `SAVE_LATEX` | `False` | Set `True` to also get a `.tex` export |


In [ ]:
import sys, importlib

# Ensure the local research_agent package is on the path
if '.' not in sys.path:
    sys.path.insert(0, '.')

# Force reimport in case this cell is re-run
for mod in list(sys.modules.keys()):
    if mod.startswith('research_agent'):
        del sys.modules[mod]

# ════════════════════════════════════════════════
#  ✏️  CONFIGURE YOUR SEARCH HERE
TOPIC      = 'Agentic AI for climate prediction'   # ← change this topic
MAX_PAPERS = 6                                      # ← 5–10 recommended
SAVE_LATEX = False                                  # ← True = also get .tex file
# ════════════════════════════════════════════════

from research_agent import run_literature_review_agent

review = run_literature_review_agent(
    topic=TOPIC,
    max_papers=MAX_PAPERS,
    save_txt=True,
    save_latex=SAVE_LATEX,
    output_dir='outputs',
)


---
## Step 5 — Download Your Review

Run this cell to download the generated `.txt` file (and `.tex` if you enabled it).


In [ ]:
import glob

output_files = sorted(glob.glob('outputs/*.txt') + glob.glob('outputs/*.tex'))

if not output_files:
    print("No output files found. Make sure save_txt=True in Step 4 and re-run.")
else:
    print(f"Found {len(output_files)} output file(s):")
    for f in output_files:
        size = os.path.getsize(f)
        print(f"  📄 {f}  ({size:,} bytes)")

    # Download the most recent file
    latest = output_files[-1]
    print(f"\nDownloading: {latest}")
    try:
        from google.colab import files
        files.download(latest)
    except ImportError:
        print(f"(Not in Colab — file saved at: {os.path.abspath(latest)})")


---
## Step 6 — Inspect Individual Pipeline Stages (Optional)

Run these cells individually to understand what happens at each step.  
Useful for debugging or learning how the agent works.


### Stage 1: Retrieve papers from arXiv

In [ ]:
import sys
if '.' not in sys.path:
    sys.path.insert(0, '.')

from research_agent.config import Config
from research_agent.arxiv_retriever import ArxivRetriever

config    = Config(max_papers=5)
retriever = ArxivRetriever(config)
papers    = retriever.search(TOPIC)

print(f"\n{'─'*60}")
print(f"Retrieved {len(papers)} papers.")
print(f"{'─'*60}")
for i, p in enumerate(papers, 1):
    print(f"\n[{i}] {p['title']}")
    print(f"     Authors : {', '.join(p['authors'][:3])}")
    print(f"     Date    : {p['date']}")
    print(f"     Abstract: {p['abstract'][:200]}...")


### Stage 2: Summarize a single paper

In [ ]:
from research_agent.summarizer import Summarizer

summarizer = Summarizer(config)
summary    = summarizer.summarize_paper(papers[0])

print(f"PAPER : {papers[0]['title']}")
print(f"{'─'*60}")
print(f"SUMMARY:\n{summary}")


### Stage 3: Build vector store and test semantic search

In [ ]:
from research_agent.arxiv_retriever import ArxivRetriever
from research_agent.vector_store import VectorStore

doc_texts   = ArxivRetriever.to_plain_text(papers)
vs          = VectorStore(config)
vs.build(doc_texts)

query  = 'deep learning atmospheric model'
chunks = vs.retrieve(query, k=2)

print(f"Query: '{query}'")
print(f"{'─'*60}")
for i, c in enumerate(chunks, 1):
    print(f"\nChunk {i}:")
    print(c[:300] + "...")


### Stage 4: Generate the full synthesis manually

In [ ]:
from research_agent.summarizer import Summarizer
from research_agent.literature_generator import LiteratureGenerator

# Summarize all papers
summarizer = Summarizer(config)
papers_with_summaries = summarizer.summarize_all(papers)

# Synthesize the review
generator   = LiteratureGenerator(config)
review_text = generator.generate(TOPIC, papers_with_summaries)

# Format and print
final = LiteratureGenerator.format_output(TOPIC, papers_with_summaries, review_text)
print(final)
